# 0412 · Modelos base, versión 2: referencias y lineal / logística

> **Versión 2 (05-11-2026, evaluación del director).** Lee `V4/MODELADO/` (0411 v2), comprueba la huella de la base (A6) y agrega referencias:
> - **C3 (regresión):** mediana de horas por zona×franja del entrenamiento y regresión de cuantil 0,5 (`QuantileRegressor`). Así un ensamble con pérdida L1 se compara con referencias de mediana, y uno con pérdida L2 con las de media.
> - **A5 (clasificación):** tasa histórica de incumplimiento de la zona×franja (sin futuro) y la regla operativa «menor margen primero».

| Modelo | Clasificación (`clf`) | Regresión (`reg`) |
| --- | --- | --- |
| `Referencia` | Prevalencia del entrenamiento, igual para todas | Media histórica de horas de su zona×franja (sin futuro) |
| `Referencia_tasa_historica` | Tasa histórica de incumplimiento de su zona×franja (sin futuro) | |
| `Referencia_menor_margen` | Menos margen hasta el vencimiento, más riesgo (solo ordena) | |
| `Referencia_mediana` | | Mediana de horas de su zona×franja en el entrenamiento |
| `Lineal` | Regresión logística | Regresión lineal (media) |
| `Lineal_mediana` | | Regresión de cuantil 0,5 (mediana) |

`Referencia_menor_margen` es un puntaje para ordenar guías, no una probabilidad: sirve para AP, AUC y alertas con capacidad fija, pero su Brier y su pérdida logarítmica no se interpretan.

**No depende de los otros cuadernos de modelos.** Solo necesita 0411 v2. No hay prueba final aquí.

In [0]:
# Si faltan librerías:
# !pip install duckdb lightgbm xgboost scikit-learn joblib

In [0]:
%pip install "duckdb<2" lightgbm xgboost shap
dbutils.library.restartPython()

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 118.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 98.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 159.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.3/547.3 kB 15.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 305.1/305.1 MB 129.1 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [shap]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## 0. Configuración

In [0]:
import os, time, warnings
import numpy as np, pandas as pd
from sklearn.linear_model import QuantileRegressor
warnings.filterwarnings("ignore")   # E1: sin avisos que impriman rutas del espacio de trabajo

# MODULOS/ debe estar junto al cuaderno, como en 0100 (MODULOS/conexion_sql.py)
from MODULOS.modelado_temporal import (cargar_preparacion, verificar_huella, pliegues_origen_movil, datos_del_pliegue,
                                       marco_predicciones, construir_modelo, predecir, tabla_metricas,
                                       mediana_por_grupo, puntaje_menor_margen)

In [0]:
PATH_DATABRICKS = "/Volumes/proyectogrado/source/source/"
CARPETA_SALIDAS  = f"{PATH_DATABRICKS}LOG_CONSOLIDADO/REVISION_2026_11/V4/"
CARPETA_MODELADO = f"{CARPETA_SALIDAS}MODELADO/"
MAX_FILAS_CUANTIL = 30_000    # submuestra para la regresión de cuantil: el solver crece casi con n² (200.000 filas tardaban horas)

## 1. Cargar la preparación de 0411

In [0]:
print("Huella de la base comprobada:", verificar_huella(CARPETA_MODELADO)[:16], "...")   # A6
CFG, df, HIST = cargar_preparacion(CARPETA_MODELADO)
SEMILLA, CATEGORICAS, VERSIONES, TAREAS = CFG["SEMILLA"], CFG["CATEGORICAS"], CFG["VERSIONES"], CFG["TAREAS"]
PLIEGUES = {t: list(pliegues_origen_movil(df, CFG["INICIOS_VALIDACION"], CFG["FIN_DESARROLLO"], t_etiqueta=te))
            for t, _, te in TAREAS}
print(f"{len(df):,} guías · versiones: {list(VERSIONES)} · bloques: {CFG['INICIOS_VALIDACION']} · preparado el {CFG['fecha_preparacion']}")

Huella de la base comprobada: dc7400e56c1f74c7 ...
1,553,980 guías · versiones: ['internas', 'internas+clima'] · bloques: ['2026-04-01', '2026-05-01', '2026-06-01', '2026-07-01', '2026-08-01'] · preparado el 2026-11-07T14:07:13.019713


## 2. Validación de origen móvil

En cada versión de variables, tarea y bloque: se arma el entrenamiento (antes del bloque) y la validación (el bloque), se calculan las referencias y se entrenan los modelos lineales. Las referencias solo usan el entrenamiento o la historia sin futuro de 0411.

In [0]:
pred_ref, pred_lin = [], []
t_ini = time.time()
for version, numericas in VERSIONES.items():
    cols = CATEGORICAS + numericas
    for tarea, y_col, _ in TAREAS:
        for ini, tr, va in PLIEGUES[tarea]:
            d, tr, va = datos_del_pliegue(df, HIST[ini], y_col, tr, va)
            Dtr, Dva = d.iloc[tr], d.iloc[va]
            Xtr, ytr, Xva, yva = Dtr[cols], Dtr[y_col], Dva[cols], Dva[y_col]
            base = marco_predicciones(version, tarea, ini, Dva, yva)
            if tarea == "clf":
                pred_ref.append(base.assign(modelo="Referencia", pred=np.full(len(Dva), ytr.mean())))
                pred_ref.append(base.assign(modelo="Referencia_tasa_historica",
                                            pred=Dva["hist_inc_zona_franja"].fillna(ytr.mean()).values))
                pred_ref.append(base.assign(modelo="Referencia_menor_margen",
                                            pred=puntaje_menor_margen(Dva["margen_sla_en_t0_h"].values)))
            else:
                pred_ref.append(base.assign(modelo="Referencia", pred=Dva["hist_h_zona_franja"].fillna(ytr.mean()).values))
                pred_ref.append(base.assign(modelo="Referencia_mediana",
                                            pred=mediana_por_grupo(ytr.values, Dtr["zona_franja"].values, Dva["zona_franja"].values)))
            # Lineal (regresión lineal o logística según la tarea)
            m = construir_modelo(tarea, "Lineal", numericas, CATEGORICAS, {}, SEMILLA).fit(Xtr, ytr)
            pred_lin.append(base.assign(modelo="Lineal", pred=predecir(m, Xva, tarea)))
            if tarea == "reg" and version == "internas":   # C3: lineal de mediana (cuantil 0,5); solo internas, que es donde se usa
                sub = np.arange(len(Xtr))
                if len(sub) > MAX_FILAS_CUANTIL:
                    sub = np.sort(np.random.default_rng(SEMILLA).choice(sub, MAX_FILAS_CUANTIL, replace=False))
                mq = construir_modelo("reg", "Lineal", numericas, CATEGORICAS, {}, SEMILLA)
                mq.set_params(modelo=QuantileRegressor(quantile=0.5, alpha=0.0, solver="highs"))
                mq.fit(Xtr.iloc[sub], ytr.iloc[sub])
                pred_lin.append(base.assign(modelo="Lineal_mediana", pred=predecir(mq, Xva, "reg")))
            print(f"{version} · {tarea} · {ini.date()} listo ({time.time() - t_ini:.0f} s)")
pred_ref, pred_lin = pd.concat(pred_ref, ignore_index=True), pd.concat(pred_lin, ignore_index=True)

internas · clf · 2026-04-01 listo (17 s)
internas · clf · 2026-05-01 listo (34 s)
internas · clf · 2026-06-01 listo (53 s)
internas · clf · 2026-07-01 listo (71 s)
internas · clf · 2026-08-01 listo (93 s)
internas · reg · 2026-04-01 listo (154 s)
internas · reg · 2026-05-01 listo (217 s)
internas · reg · 2026-06-01 listo (284 s)
internas · reg · 2026-07-01 listo (362 s)
internas · reg · 2026-08-01 listo (438 s)
internas+clima · clf · 2026-04-01 listo (460 s)
internas+clima · clf · 2026-05-01 listo (475 s)
internas+clima · clf · 2026-06-01 listo (498 s)
internas+clima · clf · 2026-07-01 listo (525 s)
internas+clima · clf · 2026-08-01 listo (556 s)
internas+clima · reg · 2026-04-01 listo (571 s)
internas+clima · reg · 2026-05-01 listo (582 s)
internas+clima · reg · 2026-06-01 listo (597 s)
internas+clima · reg · 2026-07-01 listo (618 s)
internas+clima · reg · 2026-08-01 listo (645 s)


## 3. Vista rápida

Solo para revisar los modelos base. La comparación con los ensambles se hace en 0416.

In [0]:
display(tabla_metricas(pd.concat([pred_ref, pred_lin]), ["version", "tarea", "modelo"]).round(4))

version,tarea,modelo,n,prevalencia,AP,AUC_ROC,Brier,LogLoss,MAE_h,RMSE_h,P90_error_abs_h,MAE_10pct_mas_lento_h,MAPE_pct_y>=1h,cobertura_MAPE
internas,clf,Lineal,966041,0.0765,0.2748,0.7118,0.0635,0.2406,null,null,null,null,null,null
internas,clf,Referencia,966041,0.0765,0.0747,0.4925,0.0707,0.2708,null,null,null,null,null,null
internas,clf,Referencia_menor_margen,966041,0.0765,0.1253,0.5777,0.1336,0.4453,null,null,null,null,null,null
internas,clf,Referencia_tasa_historica,966041,0.0765,0.1944,0.6211,0.066,0.2524,null,null,null,null,null,null
internas,reg,Lineal,958475,null,null,null,null,null,7.7716,31.7644,11.2713,41.1159,106.7879,0.97
internas,reg,Lineal_mediana,958475,null,null,null,null,null,6.7736,32.4294,6.9888,45.7413,58.0364,0.97
internas,reg,Referencia,958475,null,null,null,null,null,7.8891,31.8511,10.3899,41.4672,114.2549,0.97
internas,reg,Referencia_mediana,958475,null,null,null,null,null,6.8066,32.2946,6.6453,45.4749,64.2894,0.97
internas+clima,clf,Lineal,966041,0.0765,0.2943,0.7233,0.0626,0.2371,null,null,null,null,null,null
internas+clima,clf,Referencia,966041,0.0765,0.0747,0.4925,0.0707,0.2708,null,null,null,null,null,null


In [0]:
display(tabla_metricas(pd.concat([pred_ref, pred_lin]), ["version", "tarea", "modelo", "bloque"]).round(4))

version,tarea,modelo,bloque,n,prevalencia,AP,AUC_ROC,Brier,LogLoss,MAE_h,RMSE_h,P90_error_abs_h,MAE_10pct_mas_lento_h,MAPE_pct_y>=1h,cobertura_MAPE
internas,clf,Lineal,2026-04-01,193214,0.0711,0.2503,0.7137,0.0603,0.2303,null,null,null,null,null,null
internas,clf,Lineal,2026-05-01,204833,0.0857,0.299,0.7081,0.07,0.2615,null,null,null,null,null,null
internas,clf,Lineal,2026-06-01,185344,0.082,0.2785,0.7038,0.0678,0.2543,null,null,null,null,null,null
internas,clf,Lineal,2026-07-01,246075,0.0715,0.2703,0.7207,0.0597,0.228,null,null,null,null,null,null
internas,clf,Lineal,2026-08-01,136575,0.0717,0.281,0.7228,0.0595,0.2278,null,null,null,null,null,null
internas,clf,Referencia,2026-04-01,193214,0.0711,0.0711,0.5,0.0661,0.257,null,null,null,null,null,null
internas,clf,Referencia,2026-05-01,204833,0.0857,0.0857,0.5,0.0788,0.2958,null,null,null,null,null,null
internas,clf,Referencia,2026-06-01,185344,0.082,0.082,0.5,0.0754,0.2847,null,null,null,null,null,null
internas,clf,Referencia,2026-07-01,246075,0.0715,0.0715,0.5,0.0664,0.2575,null,null,null,null,null,null
internas,clf,Referencia,2026-08-01,136575,0.0717,0.0717,0.5,0.0666,0.2581,null,null,null,null,null,null


## 4. Guardar para 0416 v2

In [0]:
pred_ref.to_parquet(f"{CARPETA_MODELADO}pred_val_Referencia.parquet", index=False)
pred_lin.to_parquet(f"{CARPETA_MODELADO}pred_val_Lineal.parquet", index=False)
print("Guardado en", CARPETA_MODELADO)

Guardado en /Volumes/proyectogrado/source/source/LOG_CONSOLIDADO/REVISION_2026_11/V4/MODELADO/
